In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import pandas as pd
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

In [ ]:
DATA_DIR = Path.cwd().parent.resolve() / "data"
metadata_dir = DATA_DIR / "GF1_B02_B03_B04_B08_Mask"
metadata_path = metadata_dir / "metadata.csv"

In [ ]:
df = pd.read_csv(metadata_path)
df.head()

In [ ]:
metadata = list(metadata_dir.glob("*.tif"))
metadata

In [ ]:
def update_path(row):
    file_suffix = ".tif"
    filename = row["filename"] + file_suffix
    path = os.path.join(metadata_dir, filename)
    return path

df["path"] = df.apply(update_path, axis=1)
df = df[df["path"].apply(os.path.exists)]
df.head(10)

In [ ]:
CSV_OUT_DIR = DATA_DIR / "metadata"
IMG_OUT_DIR = DATA_DIR / "images"
LABEL_OUT_DIR = DATA_DIR / "labels"
PRED_OUT_DIR = DATA_DIR / "predictions"

In [ ]:
training_targets_txt = CSV_OUT_DIR / "training_targets.txt"
with open(training_targets_txt, 'r', encoding='utf-8') as f:
    training_targets = [line.strip() for line in f if line.strip()]
df_for_training = df[df["filename"].isin(training_targets)].copy()
df_for_training

In [ ]:
testing_targets_txt = CSV_OUT_DIR / "testing_targets.txt"
with open(testing_targets_txt, 'r', encoding='utf-8') as f:
    testing_targets = [line.strip() for line in f if line.strip()]
df_for_testing = df[df["filename"].isin(testing_targets)].copy()
df_for_testing

In [ ]:
import traceback

try:
    for i in range(len(df_for_training)):
        t = GeoTIFFTiler(df_row=df_for_training.iloc[i], is_for_training=True, display_thumbnail=True)
        t.get_chips(CSV_OUT_DIR, IMG_OUT_DIR, LABEL_OUT_DIR)
    for i in range(len(df_for_testing)):
        t = GeoTIFFTiler(df_row=df_for_testing.iloc[i], is_for_training=False, display_thumbnail=True)
        t.get_chips(CSV_OUT_DIR, IMG_OUT_DIR, LABEL_OUT_DIR)
        t.predict(PRED_OUT_DIR, fast_dev_run=False)
except Exception as e:
    traceback.print_exc()

In [ ]:
%%file cloud_helper.py
import rasterio
from tqdm import tqdm
import pandas as pd
from loguru import logger
from pathlib import Path
import numpy as np
from functools import partial
import concurrent
from typing import Dict, Any, List
import torch
from PIL import Image
import abc

from MyCloudSenseNet.benchmark.cloud_dataset import CloudDataset
from MyCloudSenseNet.benchmark.cloud_model import CloudModel

BANDS = ["B02", "B03", "B04", "B08"]
CHIP_SIZE = 512
OVERLAP_RATIO = 0.2
VALID_THRESHOLD = [0.001, 0.999]
DATA_DIR = Path.cwd().parent.resolve() / "data"
MODEL_NAME = "unet"


def mask2label(mask):
    label = np.zeros_like(mask, dtype=np.uint8)
    label[(mask >= 0) & (mask <= 50)] = 0  # background
    label[(mask >= 100) & (mask <= 200)] = 1  # shadow
    label[(mask >= 250) & (mask <= 255)] = 2  # cloud
    return label


def stretch(arr, p=2, a_min=0, a_max=1):
    lo = np.percentile(arr, p)
    hi = np.percentile(arr, 100 - p)
    return np.clip((arr - lo) / (hi - lo), a_min, a_max)


def is_label_valid(label: np.ndarray, valid_threshold: List[float] = VALID_THRESHOLD) -> bool:
    valid_pixel = np.sum((label > 0.01) & (label <= 2.0))
    total_pixel = label.size
    valid_percent = valid_pixel / total_pixel
    return valid_threshold[0] <= valid_percent <= valid_threshold[1]


class BaseGeoTIFFProcessor(metaclass=abc.ABCMeta):
    """
    抽象基类，定义GeoTIFF处理器的通用接口
    """

    def __init__(self, df_row: pd.Series, is_for_training: bool = True, display_thumbnail: bool = False):
        self.df_row = df_row
        self.info = None
        self.is_for_training = is_for_training

    @abc.abstractmethod
    def read_data(self, display_thumbnail: bool = False):
        """读取数据的抽象方法"""
        pass

    @abc.abstractmethod
    def process(self, *args, **kwargs):
        """处理数据的抽象方法"""
        pass

    @abc.abstractmethod
    def predict(self, pred_out_dir: Path = DATA_DIR / "predictions",
                fusion_method: str = "vote", fast_dev_run: bool = False):
        """预测的抽象方法"""
        pass


class LargeImageProcessor(BaseGeoTIFFProcessor):
    """
    处理大图的处理器
    """

    def __init__(self, df_row: pd.Series, is_for_training: bool = True, display_thumbnail: bool = False):
        super().__init__(df_row, is_for_training, display_thumbnail)
        self._read(display_thumbnail)

    def _read(self, display_thumbnail: bool):
        """读取大图数据"""
        with rasterio.open(self.df_row.path) as src:
            self.info = {
                "data": src.read((1, 2, 3, 4)),
                "label": mask2label(src.read((5))),
                "meta": src.meta.copy(),
                "transform": src.transform,
                "crs": src.crs,
                "height": src.height,
                "width": src.width,
            }

            if display_thumbnail:
                self._display_thumbnail(src)

    def _display_thumbnail(self, src):
        """显示缩略图"""
        import matplotlib.pyplot as plt

        max_size = 512
        h, w = src.height, src.width
        scale = min(max_size / w, max_size / h)
        new_w, new_h = int(w * scale), int(h * scale)
        thumbnail = src.read(out_shape=(src.count, new_h, new_w),
                             resampling=rasterio.enums.Resampling.bilinear)

        name = Path(self.df_row.path).stem
        b, g, r, nir, mask = thumbnail[:5]
        rgb = stretch(np.dstack((r, g, b)))

        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        logger.debug(name)
        fig.suptitle(name, fontsize=14, fontweight="bold")

        ax[0].imshow(rgb)
        ax[0].set_title("RGB Image")

        ax[1].imshow(mask)
        ax[1].set_title("Mask Image")

        plt.tight_layout()
        plt.show()

    def calculate_transform(self, x_start: int, y_start: int, x_end: int, y_end: int, chip_size: int) -> rasterio.Affine:
        """计算切片的地理变换"""
        orig_transform = self.info["transform"]
        x_min = orig_transform.xoff + x_start * orig_transform.a
        y_max = orig_transform.yoff + y_start * orig_transform.e
        x_max = orig_transform.xoff + x_end * orig_transform.a
        y_min = orig_transform.yoff + y_end * orig_transform.e

        return rasterio.transform.from_bounds(x_min, y_min, x_max, y_max, chip_size, chip_size)

    def generate_chip_windows(self, chip_size=CHIP_SIZE, overlap_ratio=OVERLAP_RATIO):
        """生成切片窗口"""
        stride = int(chip_size * (1 - overlap_ratio))
        height, width = self.info.get("height", 0), self.info.get("width", 0)
        windows = []
        chip_idx = 0

        # Row-wise
        y_start = 0
        while y_start < height:
            y_end = min(y_start + chip_size, height)

            # Column-wise
            x_start = 0
            while x_start < width:
                x_end = min(x_start + chip_size, width)

                chip_id = f"{self.df_row.filename}_chip_{chip_idx:04d}"
                windows.append({
                    "chip_idx": chip_idx,
                    "chip_id": chip_id,
                    "chip_size": chip_size,
                    "x_start": x_start,
                    "x_end": x_end,
                    "y_start": y_start,
                    "y_end": y_end,
                    "transform": self.calculate_transform(x_start, y_start, x_end, y_end, chip_size),
                })

                x_start += stride
                chip_idx += 1
            y_start += stride

        self.info["windows"] = windows

        logger.info(f"Generating overlapping windows: Total {len(windows)} chips | Overlap ratio {overlap_ratio * 100:.0f}% | Stride {stride}")

    def save_chip(self, chip_data: np.ndarray, chip_label: np.ndarray, chip_info: Dict[str, Any], output_dirs: Dict[str, Path]) -> Dict[str, str]:
        """保存切片"""
        chip_id = chip_info["chip_id"]
        chip_size = chip_info["chip_size"]
        chip_dir = output_dirs["img"] / chip_id
        chip_dir.mkdir(exist_ok=True, parents=True)
        label_out_dir = output_dirs["label"]
        label_out_dir.mkdir(exist_ok=True, parents=True)

        meta = {
            "driver": "GTiff",
            "height": chip_size,
            "width": chip_size,
            "count": 1,
            "dtype": "uint16",
            "crs": chip_info["crs"],
            "transform": chip_info["transform"],
            "compress": "deflate",
            "predictor": 2,
        }

        paths = {}
        for idx, band in enumerate(BANDS):
            path = chip_dir / f"{band}.tif"
            if not path.exists():
                with rasterio.open(path, "w", **meta) as dst:
                    dst.write(chip_data[idx], 1)
            paths[f"{band}_path"] = str(path.resolve())

        path = label_out_dir / f"{chip_id}.tif"
        if not path.exists():
            with rasterio.open(path, "w", **meta) as dst:
                dst.write(chip_label, 1)
        paths["label_path"] = str(path.resolve())
        return paths

    def get_single_chip(self, window: Dict[str, Any], full_data: np.ndarray, full_label: np.ndarray, output_dirs: Dict[str, Path]) -> Dict[str, Any]:
        """获取单个切片"""
        x1, x2 = window["x_start"], window["x_end"]
        y1, y2 = window["y_start"], window["y_end"]

        chip_label = full_label[y1:y2, x1:x2]

        if self.is_for_training and not is_label_valid(chip_label):
            return None

        chip_data = full_data[:, y1:y2, x1:x2]

        chip_info = {
            "chip_id": window["chip_id"],
            "chip_size": window["chip_size"],
            "transform": window["transform"],
            "crs": self.info["crs"],
            "x_start": x1, "x_end": x2,
            "y_start": y1, "y_end": y2
        }

        paths = self.save_chip(chip_data, chip_label, chip_info, output_dirs)

        return {
            "chip_id": window["chip_id"],
            "location": self.df_row.location,
            "datetime": self.df_row.datetime,
            "x_start": x1, "x_end": x2,
            "y_start": y1, "y_end": y2,
            **paths,
        }

    def process(self, csv_out_dir: Path, img_out_dir: Path, label_out_dir: Path, max_workers: int = 4) -> None:
        """处理大图，生成切片"""
        chip_size = CHIP_SIZE
        self.generate_chip_windows(chip_size=chip_size)
        csv_out_dir.mkdir(exist_ok=True, parents=True)
        img_out_dir.mkdir(exist_ok=True, parents=True)
        label_out_dir.mkdir(exist_ok=True, parents=True)

        filename = self.df_row.filename
        img_chip_out_dir = img_out_dir / filename
        label_chip_out_dir = label_out_dir / filename

        img_chip_out_dir.mkdir(exist_ok=True, parents=True)
        label_chip_out_dir.mkdir(exist_ok=True, parents=True)

        output_dirs = {
            "img": img_chip_out_dir,
            "label": label_chip_out_dir
        }

        metadata = []
        windows = self.info["windows"]
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            process_func = partial(
                self.get_single_chip,
                full_data=self.info["data"],
                full_label=self.info["label"],
                output_dirs=output_dirs
            )
            futures = {executor.submit(process_func, win): win for win in windows}

            for future in tqdm(concurrent.futures.as_completed(futures), total=len(windows), desc="Processing chips"):
                result = future.result()
                if result:
                    metadata.append(result)

        chip_df = pd.DataFrame(metadata)
        self.info["metadata"] = chip_df
        chip_df.to_csv(csv_out_dir / f"{filename}_metadata.csv", index=False)
        logger.info(f"Process {filename} completed! Generated {len(metadata)} valid chips.")

    def predict_chips(self, pred_out_dir: Path, fast_dev_run: bool = False):
        """预测切片"""
        logger.info("Loading model weights")
        model_weights_path: Path = Path(f"benchmark/{MODEL_NAME}/assets/cloud_model.pt")

        logger.info("Loading model")
        model = CloudModel(bands=BANDS,
                           hparams={"weights": None},
                           model_name=MODEL_NAME)
        model.load_state_dict(torch.load(model_weights_path))

        logger.info("Loading metadata")
        x_paths = self.info["metadata"]
        if fast_dev_run:
            x_paths = x_paths.head(model.num_workers * model.num_workers)
        logger.info(f"Found {len(x_paths)} chips")

        dataset = CloudDataset(x_paths=x_paths, bands=BANDS)
        dataloader = torch.utils.data.DataLoader(dataset,
                                                 batch_size=model.batch_size,
                                                 num_workers=model.num_workers,
                                                 pin_memory=False)

        logger.info("Generating predictions in batches")
        for batch_index, batch in enumerate(dataloader):
            if batch_index % 30 == 0:
                logger.debug(f"Predicting batch {batch_index} of {len(dataloader)}")
            device = next(model.parameters()).device
            x = batch["chip"].to(device)
            model.eval()
            with torch.no_grad():
                preds = model(x)
                preds = torch.argmax(preds, dim=1).cpu().numpy().astype("uint8")
            for chip_id, pred in zip(batch["chip_id"], preds):
                chip_pred_path = pred_out_dir / f"{chip_id}.tif"
                chip_pred_im = Image.fromarray(pred)
                chip_pred_im.save(chip_pred_path)

        logger.info(f"""Saved {len(list(pred_out_dir.glob("*.tif")))} predictions""")

    def predict(self, pred_out_dir: Path = DATA_DIR / "predictions", fusion_method: str = "vote", fast_dev_run: bool = False):
        """预测大图"""
        logger.info("Starting predicting chips")
        pred_label_dir = pred_out_dir / self.df_row.filename
        pred_label_dir.mkdir(exist_ok=True, parents=True)
        self.predict_chips(pred_label_dir, fast_dev_run)

        windows = self.info["windows"]
        height, width = self.info["height"], self.info["width"]
        meta, crs, transform = self.info["meta"], self.info["crs"], self.info["transform"]

        # pred_full：Used to store the restored prediction results
        # weight_full：Used to record the number of times each pixel is covered (for overlay blending)
        pred_full = np.zeros((height, width), dtype=np.float32)
        weight_full = np.zeros((height, width), dtype=np.float32)

        logger.info("Starting restoring predicted chips to a large image")
        for win in tqdm(windows, total=len(windows), desc="restoring"):
            x1, y1 = int(win["x_start"]), int(win["y_start"])
            x2, y2 = int(win["x_end"]), int(win["y_end"])
            chip_pred_path = Path(pred_label_dir) / f"{win['chip_id']}.tif"
            if not chip_pred_path.exists():
                continue
            with rasterio.open(chip_pred_path) as src:
                chip_pred = src.read(1)

            chip_h, chip_w = y2 - y1, x2 - x1
            pred_full[y1:y2, x1:x2] += chip_pred[:chip_h, :chip_w]
            weight_full[y1:y2, x1:x2] += 1

        weight_full[weight_full == 0] = 1

        if fusion_method == "average":
            pred_full = pred_full / weight_full
        elif fusion_method == "vote":
            pred_full = np.round(pred_full / weight_full).astype(np.int32)

        logger.info("Saving prediction results as GeoTIFF with geographic coordinates")
        meta.update({
            "count": 1,
            "dtype": np.uint8,
            "height": height,
            "width": width,
            "crs": crs,
            "transform": transform,
            "compress": "deflate",
            "predictor": 2,
        })

        pred_out_path = pred_out_dir / f"{self.df_row.filename}_PredictedMask.tif"
        with rasterio.open(pred_out_path, "w", **meta) as dst:
            dst.write(pred_full, 1)

        logger.info("Restoration completed!")


class ChipProcessor(BaseGeoTIFFProcessor):
    """
    处理已分片数据的处理器
    """

    def __init__(self, df_row: pd.Series, is_for_training: bool = True, display_thumbnail: bool = False):
        super().__init__(df_row, is_for_training, display_thumbnail)
        self.chip_data = []
        self.chip_labels = []
        self.chip_metadata = []
        self._read()

    def _read(self):
        """读取已分片的数据"""
        # 假设df_row包含已分片数据的信息
        # 可能是包含多个切片路径的DataFrame
        if hasattr(self.df_row, 'chip_paths') and isinstance(self.df_row.chip_paths, list):
            # 如果df_row包含chip_paths列表
            for chip_path in self.df_row.chip_paths:
                self._load_chip(chip_path)
        else:
            # 如果是单个切片文件，但已经是分片
            self._load_single_chip()

    def _load_chip(self, chip_path: str):
        """加载单个切片"""
        # 这里可以根据实际情况调整
        # 假设每个切片都有对应的标签文件
        chip_file = Path(chip_path)
        label_file = chip_file.parent.parent / "labels" / f"{chip_file.stem}.tif"

        with rasterio.open(chip_file) as src:
            chip_data = src.read()
            self.chip_data.append(chip_data)

        if label_file.exists():
            with rasterio.open(label_file) as src:
                chip_label = src.read(1)
                self.chip_labels.append(chip_label)

        # 添加元数据
        self.chip_metadata.append({
            "chip_id": chip_file.stem,
            "chip_path": str(chip_file),
            "label_path": str(label_file) if label_file.exists() else None
        })

    def _load_single_chip(self):
        """加载单一切片（如果df_row本身就是一个切片）"""
        with rasterio.open(self.df_row.path) as src:
            chip_data = src.read((1, 2, 3, 4))  # 假设前4个波段是数据
            chip_label = mask2label(src.read((5)))  # 第5个波段是标签

            self.chip_data.append(chip_data)
            self.chip_labels.append(chip_label)

            self.chip_metadata.append({
                "chip_id": self.df_row.filename,
                "chip_path": self.df_row.path,
                "label_path": self.df_row.path,  # 同一个文件，实际应用中可能不同
                "transform": src.transform,
                "crs": src.crs
            })

    def process(self, output_dir: Path = None, **kwargs):
        """处理已分片数据（这里主要是验证和组织数据）"""
        logger.info(f"Processing {len(self.chip_data)} pre-chipped images")

        # 如果需要重新组织或验证切片数据
        valid_chips = []
        for i, (data, label, meta) in enumerate(zip(self.chip_data, self.chip_labels, self.chip_metadata)):
            if self.is_for_training and not is_label_valid(label):
                logger.warning(f"Chip {meta['chip_id']} does not meet validity threshold, skipping...")
                continue
            valid_chips.append((data, label, meta))

        self.chip_data = [item[0] for item in valid_chips]
        self.chip_labels = [item[1] for item in valid_chips]
        self.chip_metadata = [item[2] for item in valid_chips]

        logger.info(f"Valid chips after filtering: {len(self.chip_data)}")

        # 保存处理后的元数据
        if output_dir:
            output_dir.mkdir(exist_ok=True, parents=True)
            metadata_df = pd.DataFrame(self.chip_metadata)
            metadata_df.to_csv(output_dir / f"{self.df_row.filename}_chips_metadata.csv", index=False)

    def predict(self, pred_out_dir: Path = DATA_DIR / "predictions",
                fusion_method: str = "vote", fast_dev_run: bool = False):
        """预测已分片数据"""
        logger.info("Loading model weights")
        model_weights_path: Path = Path(f"benchmark/{MODEL_NAME}/assets/cloud_model.pt")

        logger.info("Loading model")
        model = CloudModel(bands=BANDS,
                           hparams={"weights": None},
                           model_name=MODEL_NAME)
        model.load_state_dict(torch.load(model_weights_path))

        logger.info("Preparing chip data for prediction")
        # 构建DataFrame用于CloudDataset
        chip_df = pd.DataFrame(self.chip_metadata)

        if fast_dev_run:
            chip_df = chip_df.head(model.num_workers * model.num_workers)

        logger.info(f"Found {len(chip_df)} chips for prediction")

        dataset = CloudDataset(x_paths=chip_df, bands=BANDS)
        dataloader = torch.utils.data.DataLoader(dataset,
                                                 batch_size=model.batch_size,
                                                 num_workers=model.num_workers,
                                                 pin_memory=False)

        logger.info("Generating predictions in batches")
        pred_out_dir.mkdir(exist_ok=True, parents=True)

        for batch_index, batch in enumerate(dataloader):
            if batch_index % 30 == 0:
                logger.debug(f"Predicting batch {batch_index} of {len(dataloader)}")
            device = next(model.parameters()).device
            x = batch["chip"].to(device)
            model.eval()
            with torch.no_grad():
                preds = model(x)
                preds = torch.argmax(preds, dim=1).cpu().numpy().astype("uint8")

            for chip_id, pred in zip(batch["chip_id"], preds):
                chip_pred_path = pred_out_dir / f"{chip_id}.tif"
                chip_pred_im = Image.fromarray(pred)
                chip_pred_im.save(chip_pred_path)

        logger.info(f"""Saved {len(list(pred_out_dir.glob("*.tif")))} predictions""")

        # 如果需要融合预测结果（取决于具体需求）
        if len(self.chip_metadata) > 0 and 'transform' in self.chip_metadata[0]:
            self._fuse_predictions(pred_out_dir)

    def _fuse_predictions(self, pred_out_dir: Path):
        """融合预测结果（如果需要）"""
        # 这部分取决于具体的应用场景
        # 如果切片之间有地理坐标信息，可以考虑融合
        logger.info("Fusing predictions (if applicable)")
        # 实现融合逻辑，根据具体需求而定


class GeoTIFFProcessorFactory:
    """
    工厂类，根据输入数据类型选择合适的处理器
    """

    @staticmethod
    def create_processor(df_row: pd.Series, is_large_image: bool = None,
                         is_for_training: bool = True, display_thumbnail: bool = False) -> BaseGeoTIFFProcessor:
        """
        创建适当的处理器

        Args:
            df_row: 包含数据信息的Series
            is_large_image: 是否为大图，如果为None则自动判断
            is_for_training: 是否用于训练
            display_thumbnail: 是否显示缩略图
        """
        if is_large_image is None:
            # 自动判断：如果文件很大或者是单个大文件，则认为是大图
            file_path = Path(df_row.path)
            if file_path.exists():
                file_size = file_path.stat().st_size
                # 如果文件大于100MB，认为是大图（可根据实际情况调整阈值）
                is_large_image = file_size > 100 * 1024 * 1024
            else:
                # 如果文件不存在，根据其他特征判断
                is_large_image = False

        if is_large_image:
            return LargeImageProcessor(df_row, is_for_training, display_thumbnail)
        else:
            return ChipProcessor(df_row, is_for_training, display_thumbnail)


def process_geotiff_unified(df_row: pd.Series, is_large_image: bool = None,
                           is_for_training: bool = True, display_thumbnail: bool = False,
                           **processing_kwargs):
    """
    统一处理GeoTIFF数据的函数

    Args:
        df_row: 包含数据信息的Series
        is_large_image: 是否为大图，如果为None则自动判断
        is_for_training: 是否用于训练
        display_thumbnail: 是否显示缩略图
        **processing_kwargs: 传递给具体处理器的参数
    """
    processor = GeoTIFFProcessorFactory.create_processor(
        df_row, is_large_image, is_for_training, display_thumbnail
    )

    # 根据处理器类型执行相应的处理
    if isinstance(processor, LargeImageProcessor):
        processor.process(**processing_kwargs)
    elif isinstance(processor, ChipProcessor):
        processor.process(**processing_kwargs)

    return processor


def predict_geotiff_unified(df_row: pd.Series, is_large_image: bool = None,
                           pred_out_dir: Path = DATA_DIR / "predictions",
                           fusion_method: str = "vote", fast_dev_run: bool = False):
    """
    统一预测GeoTIFF数据的函数
    """
    processor = GeoTIFFProcessorFactory.create_processor(
        df_row, is_large_image, is_for_training=False
    )

    processor.predict(pred_out_dir=pred_out_dir,
                     fusion_method=fusion_method,
                     fast_dev_run=fast_dev_run)

    return processor